In [2]:
import pandas as pd

raw_df = pd.read_csv(
    "./data/payment_transactions_raw.csv",
    parse_dates = ["transaction_date"]
)

clean_df = raw_df.copy()
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   transaction_id     30000 non-null  object        
 1   customer_id        30000 non-null  object        
 2   merchant_id        30000 non-null  object        
 3   transaction_date   30000 non-null  datetime64[ns]
 4   payment_method     30000 non-null  object        
 5   amount             30000 non-null  float64       
 6   channel            30000 non-null  object        
 7   merchant_category  30000 non-null  object        
 8   status             30000 non-null  object        
 9   failure_reason     3663 non-null   object        
dtypes: datetime64[ns](1), float64(1), object(8)
memory usage: 2.3+ MB


### **baseline kpis**

In [12]:
print(clean_df["transaction_date"].min())
print(clean_df["transaction_date"].max())

2026-01-01 00:02:00
2026-03-30 23:58:00


In [2]:
clean_df["transaction_id"].nunique()

30000

In [3]:
clean_df["customer_id"].nunique()

7822

In [4]:
clean_df["merchant_id"].nunique()

1000

In [8]:
(clean_df["amount"].sum()).round(2)

np.float64(19562441.91)

In [10]:
(clean_df["amount"].mean()).round(2)

np.float64(652.08)

In [11]:
(clean_df["amount"].median()).round(2)

np.float64(451.92)

### **payment success rate and payment failure rate**

In [2]:
total_transactions = clean_df["transaction_id"].nunique()
successful_transactions = clean_df["status"].eq("Success").sum()
failed_transactions = clean_df["status"].eq("Failed").sum()

payment_success_rate = (successful_transactions / total_transactions).round(4)
payment_failure_rate = (failed_transactions / total_transactions).round(4)

print(successful_transactions)
print(failed_transactions)
print(payment_success_rate)
print(payment_failure_rate)

26337
3663
0.8779
0.1221


### **transactions distribution by channel**

In [11]:
transactions_by_channel = (
    clean_df
    .groupby("channel")
    .agg(
        trxs_count = ("transaction_id", "nunique")
    )
)
transactions_by_channel["perc"] = (transactions_by_channel["trxs_count"] / transactions_by_channel["trxs_count"].sum()).round(4)
transactions_by_channel.to_csv("./output/transactions_by_channel.csv")
transactions_by_channel

,trxs_count,perc
channel,,
Mobile App,13453,0.4484
POS,5993,0.1998
Web,10554,0.3518


### **transactions distribution by payment method**

In [12]:
transactions_by_payment_method = (
    clean_df
    .groupby("payment_method")
    .agg(
        trxs_count = ("transaction_id", "nunique")
    )
)
transactions_by_payment_method["perc"] = (transactions_by_payment_method["trxs_count"] / transactions_by_payment_method["trxs_count"].sum()).round(4)
transactions_by_payment_method.to_csv("./output/transactions_by_payment_method.csv")
transactions_by_payment_method

,trxs_count,perc
payment_method,,
Bank Transfer,4439,0.1480
Credit Card,9524,0.3175
Debit Card,8409,0.2803
Mobile Wallet,7628,0.2543


### **transactions distribution by merchant category**

In [13]:
transactions_by_merchant_category = (
    clean_df
    .groupby("merchant_category")
    .agg(
        trxs_count = ("transaction_id", "nunique")
    )
    .sort_values("trxs_count", ascending=False)
)
transactions_by_merchant_category["perc"] = (transactions_by_merchant_category["trxs_count"] / transactions_by_merchant_category["trxs_count"].sum()).round(4)
transactions_by_merchant_category.to_csv("./output/transactions_by_merchant_category.csv")
transactions_by_merchant_category

,trxs_count,perc
merchant_category,,
Retail,6672,0.2224
Food & Beverage,5347,0.1782
Travel,4512,0.1504
Entertainment,3590,0.1197
Utilities,3514,0.1171
Digital Services,3315,0.1105
Healthcare,3050,0.1017


### **transactions distribution by hour**

In [17]:
clean_df["hour"] = clean_df["transaction_date"].dt.hour
transactions_by_hour = (
    clean_df["hour"].value_counts().sort_index()
)
transactions_by_hour.to_csv("./output/transactions_by_hour.csv")
transactions_by_hour

hour
0     1313
1     1259
2     1223
3     1295
4     1213
5     1246
6     1263
7     1251
8     1230
9     1310
10    1247
11    1251
12    1283
13    1241
14    1230
15    1297
16    1243
17    1175
18    1245
19    1191
20    1238
21    1226
22    1257
23    1273
Name: count, dtype: int64

### **transactions distribution by week day**

In [20]:
days = [
    "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday"
]
clean_df["day_of_week"] = clean_df["transaction_date"].dt.day_name()
transactions_by_day = (
    clean_df["day_of_week"].value_counts().reindex(days)
)
transactions_by_day.to_csv("./output/transactions_by_day.csv")
transactions_by_day

day_of_week
Sunday       4476
Monday       4362
Tuesday      3991
Wednesday    3998
Thursday     4436
Friday       4350
Saturday     4387
Name: count, dtype: int64